# Spindle — All 13 Domains

Spindle ships 12 production-ready domains, each with statistically calibrated distributions sourced from authoritative industry data (NRF, CMS, CDC, BLS, HCUP, and more).

| Domain | Tables | Focus |
|---|---|---|
| retail | 9 | Customers, orders, products, returns |
| healthcare | 9 | Patients, encounters, diagnoses, claims |
| financial | 8 | Accounts, transactions, loans, fraud |
| supply_chain | 9 | Warehouses, POs, inventory, logistics |
| iot | 8 | Devices, sensors, alerts, maintenance |
| hr | 8 | Employees, departments, compensation |
| insurance | 8 | Policies, claims, underwriting |
| marketing | 8 | Campaigns, leads, conversions |
| education | 8 | Students, courses, grades, financial aid |
| real_estate | 8 | Properties, listings, transactions |
| manufacturing | 8 | Production, quality control, equipment |
| telecom | 8 | Subscribers, usage, billing, churn |

In [1]:
%pip install sqllocks-spindle --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\suref\OneDrive\VSCode\AzureClients\forge-workspace\projects\fabric-datagen\.venv-win\Scripts\python.exe -m pip install --upgrade pip


In [2]:
import time
import pandas as pd
from sqllocks_spindle import (
    Spindle,
    RetailDomain, HealthcareDomain, FinancialDomain, SupplyChainDomain,
    IoTDomain, HRDomain, InsuranceDomain, MarketingDomain,
    EducationDomain, RealEstateDomain, ManufacturingDomain, TelecomDomain,
)

spindle = Spindle()

DOMAINS = [
    ("retail",        RetailDomain()),
    ("healthcare",    HealthcareDomain()),
    ("financial",     FinancialDomain()),
    ("supply_chain",  SupplyChainDomain()),
    ("iot",           IoTDomain()),
    ("hr",            HRDomain()),
    ("insurance",     InsuranceDomain()),
    ("marketing",     MarketingDomain()),
    ("education",     EducationDomain()),
    ("real_estate",   RealEstateDomain()),
    ("manufacturing", ManufacturingDomain()),
    ("telecom",       TelecomDomain()),
]

## Generate all domains at `fabric_demo` scale

In [3]:
results = {}
bench = []
t_total = time.time()

for name, domain in DOMAINS:
    t0 = time.time()
    r = spindle.generate(domain=domain, scale="fabric_demo", seed=42)
    elapsed = time.time() - t0
    results[name] = r
    total_rows = sum(len(r[t]) for t in r.tables)
    errors = r.verify_integrity()
    status = "PASS" if not errors else f"FAIL ({len(errors)})"
    bench.append({"domain": name, "tables": len(r.tables), "rows": total_rows,
                  "elapsed_s": round(elapsed, 2), "integrity": status})
    print(f"{name:<16} {len(r.tables):>2} tables  {total_rows:>7,} rows  {elapsed:.2f}s  {status}")

print(f"\nTotal: {time.time() - t_total:.1f}s")

retail            9 tables    4,670 rows  0.23s  PASS
healthcare        9 tables    4,282 rows  0.04s  PASS
financial        10 tables    3,596 rows  0.02s  PASS
supply_chain     10 tables    5,322 rows  0.10s  PASS
iot               8 tables    3,045 rows  0.02s  PASS


hr                9 tables    1,775 rows  0.03s  PASS
insurance         9 tables    2,255 rows  0.03s  PASS


marketing        10 tables    4,440 rows  0.31s  PASS
education         9 tables    3,447 rows  0.05s  PASS
real_estate       9 tables    1,555 rows  0.04s  PASS
manufacturing     9 tables    1,515 rows  0.01s  PASS
telecom           9 tables   14,740 rows  0.10s  PASS

Total: 1.0s


In [4]:
pd.DataFrame(bench)

,domain,tables,rows,elapsed_s,integrity
0,retail,9,4670,0.23,PASS
1,healthcare,9,4282,0.04,PASS
2,financial,10,3596,0.02,PASS
3,supply_chain,10,5322,0.10,PASS
4,iot,8,3045,0.02,PASS
5,hr,9,1775,0.03,PASS
6,insurance,9,2255,0.03,PASS
7,marketing,10,4440,0.31,PASS
8,education,9,3447,0.05,PASS
9,real_estate,9,1555,0.04,PASS


## Retail — Orders

In [5]:
results["retail"]["order"].head()

,order_id,customer_id,store_id,shipping_address_id,promotion_id,order_date,status,order_total
0,1,1,101,254,143,2025-08-12 20:36:31.081033514,completed,156.40
1,2,10,46,None,None,2025-08-19 12:01:03.000000000,completed,76.20
2,3,24,1,264,None,2025-03-26 07:29:42.000000000,shipped,57.96
3,4,5,1,256,None,2025-06-20 12:43:20.628112464,completed,0.00
4,5,131,113,253,None,2024-04-25 02:52:33.000000000,shipped,263.71


## Healthcare — Diagnoses (ICD-10 codes)

In [6]:
results["healthcare"]["diagnosis"].head()

,diagnosis_id,encounter_id,icd10_code,diagnosis_type,is_chronic,description
0,1,38,N39.0,Secondary,false,"Urinary tract infection, site not specified"
1,2,102,I48.91,Primary,false,Unspecified atrial fibrillation
2,3,148,S93.401A,Admitting,true,"Sprain of unspecified ligament of right ankle,..."
3,4,23,G43.909,External Cause,false,"Migraine, unspecified, not intractable"
4,5,475,R10.9,Admitting,true,Unspecified abdominal pain


## Financial — Fraud detection

Spindle's financial domain includes a separate `fraud_flag` table linked to transactions via `transaction_id`, with flag types, risk scores, and resolutions calibrated to realistic fraud rates.

In [7]:
txn = results["financial"]["transaction"]
fraud = results["financial"]["fraud_flag"]
print(f"Transactions:  {len(txn):,}")
print(f"Fraud flags:   {len(fraud):,}")
print(f"Fraud rate:    {len(fraud) / len(txn):.2%}")
print(f"\nFlag type distribution:")
print(fraud["flag_type"].value_counts())
print(f"\nResolution distribution:")
print(fraud["resolution"].value_counts())
fraud.head()

Transactions:  1,000
Fraud flags:   20
Fraud rate:    2.00%

Flag type distribution:
flag_type
Rapid Succession    7
Unusual Amount      5
Unusual Location    3
Velocity Check      3
Account Takeover    2
Name: count, dtype: int64

Resolution distribution:
resolution
False Positive     13
Under Review        6
Confirmed Fraud     1
Name: count, dtype: int64


,flag_id,transaction_id,flag_type,risk_score,flagged_date,resolution
0,1,405,Rapid Succession,0.706746,2023-10-08 00:56:59.931061065,False Positive
1,2,342,Unusual Location,0.572859,2024-06-30 01:17:59.557729305,False Positive
2,3,506,Unusual Amount,0.587874,2022-08-10 05:25:51.268135018,False Positive
3,4,957,Rapid Succession,0.756287,2023-03-15 02:48:41.322361099,False Positive
4,5,998,Velocity Check,0.514741,2023-01-03 18:00:03.803483899,Confirmed Fraud


## IoT — Sensor readings

In [8]:
results["iot"]["reading"].head()

,reading_id,sensor_id,reading_value,reading_timestamp,quality_flag
0,1,4,24.959368,2023-12-04 10:02:48.578819292,Good
1,2,3,25.014807,2023-04-10 21:00:30.428916260,Good
2,3,1,26.565565,2024-07-15 03:56:11.665889438,Good
3,4,1,25.227791,2023-12-26 14:25:55.378042304,Good
4,5,2,25.016681,2025-10-29 21:00:58.216020074,Good


## Telecom — Churn risk

In [9]:
churn = results["telecom"]["churn_indicator"]
print(f"Churn rate: {(churn['churn_score'] >= 0.5).mean():.1%}")
print("\nChurn risk distribution:")
print(churn["risk_level"].value_counts())

Churn rate: 57.0%

Churn risk distribution:
risk_level
Low          89
Medium       54
High         43
Very High    14
Name: count, dtype: int64


## HR — Compensation by department

The HR domain stores salary data in a separate `compensation` table (normalized 3NF).  We join it to `employee` to get salary statistics per department.

In [10]:
emp = results["hr"]["employee"]
comp = results["hr"]["compensation"]
emp_comp = emp.merge(comp[["employee_id", "base_salary"]], on="employee_id", how="left")
emp_comp.groupby("department_id")["base_salary"].describe().round(0)

,count,mean,std,min,25%,50%,75%,max
department_id,,,,,,,,
1,148.0,67408.0,31781.0,28000.0,47661.0,59230.0,82087.0,212342.0
2,44.0,72966.0,35180.0,28000.0,43567.0,66159.0,105123.0,138856.0
3,28.0,90074.0,50256.0,28000.0,48513.0,82489.0,113227.0,220095.0
4,10.0,63080.0,32332.0,28000.0,37895.0,61232.0,66055.0,131268.0
5,16.0,65791.0,42628.0,28000.0,40448.0,45238.0,86860.0,176192.0
6,3.0,76887.0,26106.0,48153.0,65758.0,83364.0,91254.0,99145.0
7,4.0,48994.0,11335.0,36624.0,40816.0,49774.0,57952.0,59807.0
8,2.0,50624.0,17534.0,38225.0,44424.0,50624.0,56823.0,63022.0
9,5.0,63205.0,48877.0,28000.0,38833.0,45805.0,54486.0,148900.0


## Marketing — Lead funnel

In [11]:
leads = results["marketing"]["lead"]
print("Lead status distribution:")
print(leads["status"].value_counts(normalize=True).mul(100).round(1).rename("pct %"))

Lead status distribution:
status
Contacted      31.5
Converted      22.0
Qualified      18.5
New            15.0
Unqualified    13.0
Name: pct %, dtype: float64
